# Notebook 04: Limpieza, Transformación y Normalización Canónica (Capa Silver)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Bronze $\rightarrow$ Silver  
**Estándar**: DAMA-DMBOK 2 / Clean Architecture / SWEBOK v4  

---
### Objetivos de la Capa Silver
1. **Sanitización Tipográfica**: Eliminación de acentos diacríticos (Unicode NFKD), colapso de espacios dobles y unificación a mayúsculas.
2. **Imputación Controlada**: Tratamiento determinístico de valores nulos (`SIN_VARIEDAD_ESPECIFICADA`).
3. **Tipado Estricto**: Coerción de fechas a `datetime64`, identificadores a `string`, y métricas a `float64` / `int64`.
4. **Deduplicación Temporal y Llaves Compuestas**: Consolidación sobre llaves primarias de negocio.
5. **Persistencia Parquet Snappy**: Almacenamiento optimizado y cálculo de huella criptográfica SHA-256 en `data/CLEANED/`.

In [1]:
# Celda 1: Setup e Importaciones de Módulos de Limpieza
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.ingestion.landing_manager import LandingManager
from src.cleaning.transformers import (
    sanitize_text,
    cast_datatypes,
    handle_missing_values,
    deduplicate_dataset,
)
from src.cleaning.cleaner_pipeline import CleanerPipeline

logger = get_logger("notebooks.04_cleaning")
landing = LandingManager()
cleaner = CleanerPipeline()
print(f"CleanerPipeline inicializado. Directorio Silver: {cleaner.cleaned_dir}")

### 1. Limpieza y Normalización de Evaluaciones Agropecuarias (EVA)

In [2]:
# Celda 2: Carga y saneamiento de datos agrícolas EVA
eva_path = settings.RAW_DIR / "upra" / "evaluaciones_agropecuarias_eva.parquet"
if eva_path.exists():
    df_eva_raw = pd.read_parquet(eva_path)
    df_eva_silver = cleaner.clean_eva_dataset(df_eva_raw)
    print(f"EVA Raw Filas: {len(df_eva_raw)} -> EVA Silver Filas: {len(df_eva_silver)}")
    display(df_eva_silver.head(3))
else:
    print(f"Aviso: Dataset {eva_path.name} no encontrado en Bronze. Se generará muestra de validación.")
    df_eva_mock = pd.DataFrame({
        'c_d_dep': ['05'], 'c_d_mun': ['05001'], 'departamento': ['Antioquia'],
        'municipio': ['Medellín'], 'cultivo': ['Café'], 'desagregaci_n_regional_y': [None],
        'a_o': ['2023'], 'rea_sembrada_ha': ['150.0'], 'rea_cosechada_ha': ['145.0'],
        'producci_n_t': ['320.0'], 'rendimiento_t_ha': ['2.21']
    })
    df_eva_silver = cleaner.clean_eva_dataset(df_eva_mock)
    display(df_eva_silver)

### 2. Saneamiento de Series Meteorológicas de IDEAM

In [3]:
# Celda 3: Normalización y tipado de variables meteorológicas (IDEAM)
ideam_path = settings.RAW_DIR / "ideam" / "precipitacion_pluviometrica.parquet"
if ideam_path.exists():
    df_ideam_raw = pd.read_parquet(ideam_path)
    df_ideam_silver = cleaner.clean_ideam_dataset(df_ideam_raw)
    print(f"IDEAM Raw Filas: {len(df_ideam_raw)} -> IDEAM Silver Filas: {len(df_ideam_silver)}")
    display(df_ideam_silver.head(3))
else:
    print(f"Aviso: Archivo {ideam_path.name} no encontrado en Bronze. Verificando pipeline con mock.")
    df_ideam_mock = pd.DataFrame({
        'codigoestacion': ['21206990'], 'nombreestacion': ['El Salado'],
        'fechaobservacion': ['2023-05-15 07:00:00'], 'valorobservado': ['12.5'],
        'descripcionsensor': ['Precipitación Pluviómetro']
    })
    df_ideam_silver = cleaner.clean_ideam_dataset(df_ideam_mock)
    display(df_ideam_silver)

### 3. Normalización de Series de Precios e Insumos SIPSA / DANE

In [4]:
# Celda 4: Saneamiento de insumos y precios mayoristas SIPSA
sipsa_path = settings.RAW_DIR / "dane" / "insumos_fertilizantes_sipsa.parquet"
if sipsa_path.exists():
    df_sipsa_raw = pd.read_parquet(sipsa_path)
    df_sipsa_silver = cleaner.clean_sipsa_dataset(df_sipsa_raw)
    print(f"SIPSA Raw Filas: {len(df_sipsa_raw)} -> SIPSA Silver Filas: {len(df_sipsa_silver)}")
    display(df_sipsa_silver.head(3))
else:
    print(f"Aviso: Archivo {sipsa_path.name} no encontrado en Bronze. Validando con muestra.")
    df_sipsa_mock = pd.DataFrame({
        'cod_municipio': ['11001'], 'municipio': ['Bogotá, D.C.'],
        'articulo': ['Urea 46%'], 'preciopromedio': ['145000.0'],
        'anio': ['2023'], 'mes': ['6']
    })
    df_sipsa_silver = cleaner.clean_sipsa_dataset(df_sipsa_mock)
    display(df_sipsa_silver)

### 4. Orquestación y Persistencia Masiva hacia la Capa Silver (`data/CLEANED/`)

In [5]:
# Celda 5: Procesamiento sistemático de datasets de Bronze a Silver
silver_catalog = []
raw_files = list(settings.RAW_DIR.rglob("*.parquet"))
print(f"Archivos Parquet identificados en Bronze: {len(raw_files)}")

for fpath in raw_files:
    category = fpath.parent.name
    stem = fpath.stem
    df_raw = pd.read_parquet(fpath)
    
    cleaner_type = "generic"
    if "eva" in stem or category == "upra":
        cleaner_type = "eva"
    elif "ideam" in category or any(w in stem for w in ["precipitacion", "temperatura", "radiacion"]):
        cleaner_type = "ideam"
    elif "sipsa" in stem or category == "dane":
        cleaner_type = "sipsa"
        
    summary = cleaner.process_and_save(df_raw, dataset_name=f"{stem}_silver", cleaner_type=cleaner_type)
    silver_catalog.append(summary)
    print(f"[SILVER] {stem} -> Filas: {summary['final_rows']} | SHA-256: {summary['sha256'][:12]}...")

### 5. Resumen de Calidad y Trazabilidad Criptográfica de la Capa Silver

In [6]:
# Celda 6: Resumen consolidado de la capa Silver
if silver_catalog:
    df_summary = pd.DataFrame(silver_catalog)
    display(df_summary[["dataset_name", "initial_rows", "final_rows", "removed_duplicates", "file_size_bytes", "sha256"]])
else:
    print("No se procesaron archivos en este entorno de demostración.")